<a href="https://colab.research.google.com/github/e11106013/KG/blob/main/cantonese_mandarin_translation_finetune_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cantonese → Mandarin Translation Fine-Tuning
This Colab notebook demonstrates how to fine-tune `google/mt5-small` on the [botisan-ai/cantonese-mandarin-translations](https://huggingface.co/datasets/botisan-ai/cantonese-mandarin-translations) dataset using Hugging Face Transformers.

In [ ]:
# @title 📦 安裝必要套件（隱藏輸出）

%%capture
!pip install -U transformers datasets evaluate sacrebleu



In [ ]:
# @title 📚 載入資料集
from datasets import load_dataset

dataset = load_dataset("botisan-ai/cantonese-mandarin-translations")

# @title 🔍 解析資料格式（展平 translation 欄位）
def flatten_translation(example):
    return {
        "source": example["translation"]["yue"],
        "target": example["translation"]["zh"]
    }

dataset = dataset.map(flatten_translation)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/3.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

In [ ]:
# @title 🔧 設定 Tokenizer 與模型（預設 mT5-small）
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

max_input_length = 128
max_target_length = 128

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# @title 粵語 → 普通話 前處理
def preprocess(example):
    source = "translate Cantonese to Mandarin: " + example["source"]
    inputs = tokenizer(source, max_length=max_input_length, truncation=True, padding="max_length")
    targets = tokenizer(example["target"], max_length=max_target_length, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset["train"].map(preprocess, batched=False)

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

In [ ]:
# @title 普通話 → 粵語 反向前處理
def preprocess_reverse(example):
    source = "translate Mandarin to Cantonese: " + example["target"]
    inputs = tokenizer(source, max_length=max_input_length, truncation=True, padding="max_length")
    targets = tokenizer(example["source"], max_length=max_target_length, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset_reverse = dataset["train"].map(preprocess_reverse, batched=False)


Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

In [ ]:
# @title 🧪 BLEU 評估函數
import evaluate
import numpy as np

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])


In [ ]:
# @title ⚙️ 訓練參數設定
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    #evaluation_strategy="epoch", # 每個 epoch 評估
    do_eval = False ,  # 完全不要評估的話
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    seed=42,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    logging_steps=100,
    lr_scheduler_type="linear",
    optim="adamw_torch"
)

In [ ]:
# @title 🏋️‍♂️ 建立 Trainer 並訓練
# wandb API key https://wandb.ai/authorize

from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-1087892457.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: johnwu0113 (ttuainlpkg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,0.000000
200,0.000000
300,0.000000
400,0.000000
500,0.000000
600,0.000000
700,0.000000
800,0.000000
900,0.000000
1000,0.000000


In [ ]:
!pip install huggingface_hub -q

In [ ]:
# @title 💾  Hugging Face Hub 登入並推送模型（可選）

# huggingface API

from huggingface_hub import notebook_login
notebook_login()

model.push_to_hub("your-username/mt5-cantonese-to-mandarin")
tokenizer.push_to_hub("your-username/mt5-cantonese-to-mandarin")

In [ ]:
# Load model directly
#from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

#tokenizer = AutoTokenizer.from_pretrained("botisan-ai/mt5-translate-zh-yue")
#model = AutoModelForSeq2SeqLM.from_pretrained("botisan-ai/mt5-translate-zh-yue")